In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from functools import partial
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
    T5Tokenizer, T5ForConditionalGeneration,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)
from evaluate import load
import os

### Файнтьюн БЕРТА

In [ ]:
data_train = pd.read_csv("./data/in_domain_train.csv")
data_test = pd.read_csv("./data/in_domain_dev.csv")

In [ ]:
data_train

In [ ]:
data_test

In [ ]:
data_train, data_val = train_test_split(data_train, test_size=0.2, random_state=42, stratify=data_train["acceptable"])

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(data_train.reset_index(drop=True)),
    "val": Dataset.from_pandas(data_val.reset_index(drop=True)),
    "test": Dataset.from_pandas(data_test.reset_index(drop=True)),
})

In [ ]:
dataset

In [ ]:
accuracy_metric = load("accuracy")
mcc_metric = load("matthews_correlation")

In [ ]:
model_name = "sberbank-ai/ruRoberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess(examples, tokenizer):
    result = tokenizer(examples["sentence"], padding=False, truncation=True, max_length=256)
    if "acceptable" in examples:
        result["label"] = examples["acceptable"]
    return result

In [ ]:
tokenized = dataset.map(partial(preprocess, tokenizer=tokenizer), batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
tokenized

In [ ]:
tokenized['train'][0]["attention_mask"]

In [ ]:
tokenized['val'][0]["attention_mask"]

In [ ]:
tokenized['test'][0]["attention_mask"]

In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=p.label_ids)
    mcc = mcc_metric.compute(predictions=preds, references=p.label_ids)
    return {"accuracy": acc["accuracy"], "mcc": mcc["matthews_correlation"]}

In [ ]:
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2,
    # hidden_dropout_prob=0.2,       # не 0.3
    # attention_probs_dropout_prob=0.2,
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./rubert_checkpoints",
    eval_strategy="epoch",
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    num_train_epochs=10,
    warmup_ratio=0.2,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_mcc",
    fp16=False,
    report_to="none",
)

In [ ]:
import torch
from collections import Counter

counts = Counter(tokenized["train"]["label"])
total = sum(counts.values())
class_weights = torch.tensor([total / (2 * counts[0]), total / (2 * counts[1])], dtype=torch.float32)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import EarlyStoppingCallback

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["val"],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()
rubert_results = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
print("RuBERT test results:", rubert_results)

loss - высокий, но другие метрики вроде допустимые: accuracy точно - 84, MCC - как из примера учебного, дозволенное.
На loss в таком случае не стоит обращаться внимания? Так как Loss считает потери по вероятности предсказанного слова, когда как MCC оценивает классы предложений.

### Реализация few-/zero-shot с GPT-3

In [ ]:
model_name = "ai-forever/rugpt3medium_based_on_gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
model.eval()

In [ ]:
### несколько вариантов затравок

PROMPT_TEMPLATES = [
    # Вариант 1
    "Предложение: \"{sentence}\"\nЭто предложение грамматически правильное? Ответ:",
    # Вариант 2
    "Определи, является ли предложение грамматически корректным.\nПредложение: \"{sentence}\"\nОтвет:",
    # Вариант 3
    "\"{sentence}\"\nПриемлемо ли это предложение с точки зрения русского языка?",
]

POS_WORD = "да"
NEG_WORD = "нет"

In [ ]:
def get_few_shot_examples(n, data_train):
    pos = data_train[data_train["acceptable"] == 1].sample(n // 2 + n % 2, random_state=42)
    neg = data_train[data_train["acceptable"] == 0].sample(n // 2, random_state=42)
    examples = pd.concat([pos, neg]).sample(frac=1, random_state=42)
    return examples

In [ ]:
def build_prompt(sentence, template, few_shot_df=None):
    prefix = ""
    if few_shot_df is not None and len(few_shot_df) > 0:
        parts = []
        for _, row in few_shot_df.iterrows():
            answer = POS_WORD if row["acceptable"] == 1 else NEG_WORD
            parts.append(f"Предложение: \"{row['sentence']}\"\nОтвет: {answer}")
        prefix = "\n\n".join(parts) + "\n\n"
    return prefix + template.format(sentence=sentence)

In [ ]:
# empty = tokenizer(" ", return_tensors="pt").to(model.device)
# with torch.no_grad():
#     prior_log_probs = torch.log_softmax(model(**empty).logits[0, -1], dim=-1)

def predict_single(sentence, template, few_shot_df=None):
    prompt = build_prompt(sentence, template, few_shot_df)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    
    if POS_WORD in generated:
        return 1
    elif NEG_WORD in generated:
        return 0
    else:
        # если модель сгенерировала что-то другое — смотрим логиты
        with torch.no_grad():
            logits = model(**inputs).logits[0, -1]
        pos_id = tokenizer.encode(POS_WORD, add_special_tokens=False)[0]
        neg_id = tokenizer.encode(NEG_WORD, add_special_tokens=False)[0]
        return 1 if logits[pos_id] > logits[neg_id] else 0

In [ ]:
test_sentences = data_test["sentence"].tolist()
test_labels = data_test["acceptable"].tolist()

In [ ]:
results_gpt = {}
for n_shot in [0, 1, 2, 4]:
    few_shot_df = get_few_shot_examples(n_shot, data_train) if n_shot > 0 else None
    for t_idx, template in enumerate(PROMPT_TEMPLATES):
        preds = [predict_single(s, template, few_shot_df) for s in test_sentences]
        acc = accuracy_metric.compute(predictions=preds, references=test_labels)
        mcc = mcc_metric.compute(predictions=preds, references=test_labels)
        key = f"n_shot={n_shot}_template={t_idx}"
        results_gpt[key] = {"accuracy": acc["accuracy"], "mcc": mcc["matthews_correlation"]}
        print(f"{key}: acc={acc['accuracy']:.4f}, mcc={mcc['matthews_correlation']:.4f}")

### T5

In [ ]:
MODEL_TO_HUB_NAME = {
    "t5-base": "sberbank-ai/ruT5-base",
    "t5-large": "sberbank-ai/ruT5-large",
}

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_TO_HUB_NAME["t5-large"])